# CertCF top-$k$ heuristic ablation

This notebook compares strict nearest-anchor prefixes $k=1,\ldots,7$ with the exact exhaustive optimum over the same certified atlas. It reports validity separately from optimality recovery.

## Protocol and artifact validation

The experiment uses the deterministic 32-dimensional synthetic dataset, the trained `depth_03_width_128` classifier, 10,000 balanced training points, 500 randomly sampled anchors per predicted class, and the same 1,000 balanced test queries. Atlas construction and projection both use $L_1$. Sparsity refinement is disabled so that $D_k$ is the minimum $L_1$ projection distance.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import beta, fisher_exact, wilcoxon

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'topk_heuristic_ablation'
queries = pd.read_parquet(RESULTS / 'topk_queries.parquet')
summary = pd.read_parquet(RESULTS / 'topk_summary.parquet')
ranks = pd.read_parquet(RESULTS / 'topk_minimal_k.parquet')
build = json.loads((RESULTS / 'build.json').read_text())

expected_k = list(range(1, 8))
assert sorted(queries['k'].unique().tolist()) == expected_k
assert queries[['query_position', 'k']].drop_duplicates().shape[0] == 1000 * 7
assert queries['query_idx'].nunique() == 1000
print(f"Validated {len(queries):,} query-k rows and {len(ranks):,} queries.")
print(f"Atlas: {build['certified_region_count']:,} regions, build time {build['build_wall_time_s']:.1f} s.")

## Per-$k$ results

In [ ]:
columns = [
    'k', 'strict_success_rate', 'exact_recovery_rate',
    'within_1pct_rate', 'within_5pct_rate',
    'gap_absolute_mean', 'gap_absolute_p95', 'gap_absolute_max',
    'strict_runtime_ms_mean', 'strict_projections_mean',
    'miss_probability_upper_confidence',
]
summary[columns].style.format({
    'strict_success_rate': '{:.1%}',
    'exact_recovery_rate': '{:.1%}',
    'within_1pct_rate': '{:.1%}',
    'within_5pct_rate': '{:.1%}',
    'gap_absolute_mean': '{:.4f}',
    'gap_absolute_p95': '{:.4f}',
    'gap_absolute_max': '{:.4f}',
    'strict_runtime_ms_mean': '{:.2f}',
    'strict_projections_mean': '{:.2f}',
    'miss_probability_upper_confidence': '{:.2%}',
})

## Recovery probability and approximation tolerance

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(summary['k'], 100 * summary['exact_recovery_rate'], marker='o', label='Exact atlas optimum')
ax.plot(summary['k'], 100 * summary['within_1pct_rate'], marker='s', label='Within 1%')
ax.plot(summary['k'], 100 * summary['within_5pct_rate'], marker='^', label='Within 5%')
ax.set(xlabel='Number of nearest anchors k', ylabel='Queries (%)', xticks=summary['k'])
ax.set_ylim(0, 101)
ax.grid(alpha=.25)
ax.legend(frameon=False)
sns.despine()
fig.tight_layout()

## Optimality gaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].plot(summary['k'], summary['gap_absolute_mean'], marker='o', label='Mean')
axes[0].plot(summary['k'], summary['gap_absolute_p95'], marker='s', label='95th percentile')
axes[0].plot(summary['k'], summary['gap_absolute_max'], marker='^', label='Maximum')
axes[0].set(xlabel='k', ylabel='$D_k-D^*$', xticks=summary['k'])
axes[0].legend(frameon=False)
axes[0].grid(alpha=.25)

finite_gaps = queries.loc[np.isfinite(queries['gap_relative'])].copy()
sns.ecdfplot(data=finite_gaps, x='gap_relative', hue='k', palette='viridis', ax=axes[1])
axes[1].set(xlabel='Relative excess $(D_k-D^*)/D^*$', ylabel='Empirical CDF')
axes[1].grid(alpha=.25)
sns.despine()
fig.tight_layout()

## Query cost

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(summary['k'], summary['strict_runtime_ms_mean'], marker='o', label='Strict top-k')
ax.axhline(summary['exact_runtime_ms_mean'].iloc[0], color='black', linestyle='--', label='Exhaustive atlas')
ax.set(xlabel='Number of nearest anchors k', ylabel='Mean query time (ms)', xticks=summary['k'])
ax.grid(alpha=.25)
ax.legend(frameon=False)
sns.despine()
fig.tight_layout()

## Minimum $k$ required by each query

In [ ]:
counts = ranks['recovery_rank_censored'].value_counts().sort_index()
labels = [str(int(value)) if value <= 7 else '>7' for value in counts.index]
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.bar(labels, counts.values, color=sns.color_palette('viridis', len(counts)))
ax.set(xlabel='Minimum k for exact recovery', ylabel='Queries')
sns.despine()
fig.tight_layout()

## Empirical probabilities and finite-sample bounds

For each $k$, the miss indicator is $Z_i^{(k)}=\mathbf{1}\{D_k(X_i)>D^*(X_i)+\eta\}$. The table below reports the empirical event probability, an exact two-sided 95% Clopper--Pearson interval, and an exact one-sided 95% upper confidence bound. The same calculation is repeated for exact recovery and the 1% and 5% approximation tolerances.

In [ ]:
CONFIDENCE = 0.95

def clopper_pearson(events, trials, confidence=CONFIDENCE):
    alpha = 1.0 - confidence
    lower = 0.0 if events == 0 else beta.ppf(alpha / 2, events, trials - events + 1)
    upper = 1.0 if events == trials else beta.ppf(1 - alpha / 2, events + 1, trials - events)
    upper_one_sided = 1.0 if events == trials else beta.ppf(confidence, events + 1, trials - events)
    return float(lower), float(upper), float(upper_one_sided)

event_columns = {
    'Exact optimum missed': ~queries['exact_recovery'],
    'More than 1% excess': ~queries['within_1pct'],
    'More than 5% excess': ~queries['within_5pct'],
}
probability_records = []
for event_name, event_values in event_columns.items():
    for k in expected_k:
        mask = queries['k'].eq(k)
        events = int(event_values.loc[mask].sum())
        trials = int(mask.sum())
        lower, upper, upper_one_sided = clopper_pearson(events, trials)
        probability_records.append({
            'event': event_name, 'k': k, 'events': events, 'queries': trials,
            'probability': events / trials, 'ci95_lower': lower, 'ci95_upper': upper,
            'upper_95_one_sided': upper_one_sided,
        })
probability_bounds = pd.DataFrame(probability_records)

exact_probability_bounds = probability_bounds.query("event == 'Exact optimum missed'").copy()
display(exact_probability_bounds.style.format({
    'probability': '{:.2%}', 'ci95_lower': '{:.2%}', 'ci95_upper': '{:.2%}',
    'upper_95_one_sided': '{:.2%}',
}))

fig, ax = plt.subplots(figsize=(6.6, 4.0))
x = exact_probability_bounds['k'].to_numpy()
p = exact_probability_bounds['probability'].to_numpy()
lower = exact_probability_bounds['ci95_lower'].to_numpy()
upper = exact_probability_bounds['ci95_upper'].to_numpy()
ax.errorbar(x, p, yerr=[p - lower, upper - p], marker='o', capsize=4, label='Empirical probability and 95% CI')
ax.plot(x, exact_probability_bounds['upper_95_one_sided'], marker='s', linestyle='--', label='One-sided 95% upper bound')
ax.set(xlabel='Number of nearest anchors k', ylabel='Probability of missing $D^*$', xticks=x)
ax.set_yscale('log')
ax.grid(alpha=.25, which='both')
ax.legend(frameon=False)
sns.despine()
fig.tight_layout()

### Selecting $k$ from an upper miss-probability bound

The next table chooses the smallest tested $k$ whose one-sided 95% upper confidence bound is no larger than a prescribed failure probability. `>7` means that the current sweep is insufficient for that claim.

In [ ]:
threshold_records = []
for event_name in probability_bounds['event'].unique():
    event_frame = probability_bounds.query('event == @event_name').sort_values('k')
    for alpha in [0.10, 0.05, 0.025, 0.01]:
        eligible = event_frame.loc[event_frame['upper_95_one_sided'] <= alpha, 'k']
        threshold_records.append({
            'event': event_name, 'required upper bound': alpha,
            'smallest tested k': int(eligible.iloc[0]) if len(eligible) else '>7',
        })
thresholds = pd.DataFrame(threshold_records)
thresholds.pivot(index='required upper bound', columns='event', values='smallest tested k')

## Gap statistics and bootstrap uncertainty

Unconditional gap summaries include the exact recoveries, whose gap is zero. Conditional summaries use only misses and therefore expose the magnitude of the errors hidden by zero medians and high quantiles. Bootstrap intervals resample complete queries and quantify uncertainty in the mean unconditional gap.

In [ ]:
def bootstrap_mean_interval(values, confidence=0.95, n_resamples=5000, seed=42):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = np.empty(n_resamples)
    chunk = 250
    for start in range(0, n_resamples, chunk):
        size = min(chunk, n_resamples - start)
        draws = rng.choice(values, size=(size, len(values)), replace=True)
        means[start:start + size] = draws.mean(axis=1)
    tail = (1.0 - confidence) / 2
    return tuple(np.quantile(means, [tail, 1.0 - tail]))

gap_records = []
for k, group in queries.groupby('k', sort=True):
    gaps = group['gap_absolute'].to_numpy(float)
    misses = group.loc[~group['exact_recovery'], 'gap_absolute'].to_numpy(float)
    ci_lower, ci_upper = bootstrap_mean_interval(gaps, seed=42 + int(k))
    gap_records.append({
        'k': int(k), 'misses': len(misses), 'mean gap': gaps.mean(),
        'mean gap CI lower': ci_lower, 'mean gap CI upper': ci_upper,
        'conditional mean gap': misses.mean() if len(misses) else 0.0,
        'conditional median gap': np.median(misses) if len(misses) else 0.0,
        'conditional p95 gap': np.quantile(misses, .95) if len(misses) else 0.0,
        'maximum gap': gaps.max(),
    })
gap_statistics = pd.DataFrame(gap_records)
display(gap_statistics.style.format({column: '{:.4f}' for column in gap_statistics.columns if column not in {'k', 'misses'}}))

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.0))
axes[0].plot(gap_statistics['k'], gap_statistics['mean gap'], marker='o', label='Mean, all queries')
axes[0].fill_between(gap_statistics['k'], gap_statistics['mean gap CI lower'], gap_statistics['mean gap CI upper'], alpha=.2, label='Bootstrap 95% CI')
axes[0].plot(gap_statistics['k'], gap_statistics['conditional mean gap'], marker='s', label='Mean conditional on a miss')
axes[0].set(xlabel='k', ylabel='$L_1$ optimality gap', xticks=expected_k)
axes[0].legend(frameon=False)
axes[0].grid(alpha=.25)

miss_rows = queries.loc[~queries['exact_recovery']].copy()
sns.boxplot(data=miss_rows, x='k', y='gap_absolute', showfliers=True, color='#5AAE9A', ax=axes[1])
axes[1].set(xlabel='k', ylabel='$L_1$ gap conditional on a miss')
axes[1].grid(alpha=.2, axis='y')
sns.despine()
fig.tight_layout()

## Paired incremental comparisons

Every query is evaluated at every $k$, so adjacent settings should be compared pairwise. The reduction $D_{k-1}-D_k$ is non-negative by construction. We report how many queries improve, how many newly recover the exact optimum, a bootstrap interval for the mean reduction, and a one-sided Wilcoxon signed-rank test. The $p$-value describes variation across these queries only and should not be interpreted as evidence across datasets, models, or training seeds.

In [ ]:
distance_wide = queries.pivot(index='query_idx', columns='k', values='strict_distance')
recovery_wide = queries.pivot(index='query_idx', columns='k', values='exact_recovery').astype(bool)
runtime_wide = queries.pivot(index='query_idx', columns='k', values='strict_runtime_ms')
paired_records = []
for k in expected_k[1:]:
    reduction = (distance_wide[k - 1] - distance_wide[k]).to_numpy(dtype=float, copy=True)
    reduction[np.abs(reduction) < 1e-9] = 0.0
    improved = reduction > 1e-5
    newly_exact = (~recovery_wide[k - 1] & recovery_wide[k]).sum()
    ci_lower, ci_upper = bootstrap_mean_interval(reduction, seed=100 + k)
    statistic, p_value = wilcoxon(reduction, alternative='greater', zero_method='pratt')
    paired_records.append({
        'comparison': f'{k - 1} to {k}', 'queries improved': int(improved.sum()),
        'new exact recoveries': int(newly_exact), 'mean L1 reduction': reduction.mean(),
        'mean reduction CI lower': ci_lower, 'mean reduction CI upper': ci_upper,
        'maximum reduction': reduction.max(),
        'mean added time (ms)': (runtime_wide[k] - runtime_wide[k - 1]).mean(),
        'Wilcoxon p-value': p_value,
    })
paired_comparisons = pd.DataFrame(paired_records)
paired_comparisons.style.format({
    'mean L1 reduction': '{:.5f}', 'mean reduction CI lower': '{:.5f}',
    'mean reduction CI upper': '{:.5f}', 'maximum reduction': '{:.5f}',
    'mean added time (ms)': '{:.2f}', 'Wilcoxon p-value': '{:.3g}',
})

## Stratification by target class

The dataset and query set are balanced, but the heuristic may behave differently for the two target preimages. Exact binomial intervals and Fisher tests make this asymmetry visible. Holm correction is applied across the seven class comparisons.

In [ ]:
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted_sorted = np.maximum.accumulate((len(p_values) - np.arange(len(p_values))) * p_values[order])
    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted

class_records = []
fisher_p_values = []
for k in expected_k:
    per_class = []
    for target in [0, 1]:
        values = queries.loc[queries['k'].eq(k) & queries['target_class'].eq(target), 'exact_recovery']
        successes, trials = int(values.sum()), int(len(values))
        lower, upper, _ = clopper_pearson(successes, trials)
        per_class.append((successes, trials, lower, upper))
    table = [[per_class[0][0], per_class[0][1] - per_class[0][0]], [per_class[1][0], per_class[1][1] - per_class[1][0]]]
    _, p_value = fisher_exact(table)
    fisher_p_values.append(p_value)
    class_records.append({
        'k': k, 'target 0 recovery': per_class[0][0] / per_class[0][1],
        'target 1 recovery': per_class[1][0] / per_class[1][1],
        'risk difference (1 - 0)': per_class[1][0] / per_class[1][1] - per_class[0][0] / per_class[0][1],
        'target 0 lower': per_class[0][2], 'target 0 upper': per_class[0][3],
        'target 1 lower': per_class[1][2], 'target 1 upper': per_class[1][3],
        'Fisher p-value': p_value,
    })
class_comparison = pd.DataFrame(class_records)
class_comparison['Holm-adjusted p-value'] = holm_adjust(fisher_p_values)
display(class_comparison[['k', 'target 0 recovery', 'target 1 recovery', 'risk difference (1 - 0)', 'Fisher p-value', 'Holm-adjusted p-value']].style.format({
    'target 0 recovery': '{:.1%}', 'target 1 recovery': '{:.1%}',
    'risk difference (1 - 0)': '{:+.1%}', 'Fisher p-value': '{:.3g}',
    'Holm-adjusted p-value': '{:.3g}',
}))

fig, ax = plt.subplots(figsize=(6.6, 4.0))
for target, color in [(0, '#D55E00'), (1, '#0072B2')]:
    rate = class_comparison[f'target {target} recovery'].to_numpy()
    lower = class_comparison[f'target {target} lower'].to_numpy()
    upper = class_comparison[f'target {target} upper'].to_numpy()
    ax.errorbar(expected_k, rate, yerr=[rate - lower, upper - rate], marker='o', capsize=3, label=f'Target class {target}', color=color)
ax.set(xlabel='k', ylabel='Exact-recovery probability', xticks=expected_k, ylim=(0.7, 1.01))
ax.grid(alpha=.25)
ax.legend(frameon=False)
sns.despine()
fig.tight_layout()

## Cost--quality trade-off

The exhaustive reference solves about 497 projections and takes roughly 1.49 s per query on this machine. The table reports the top-$k$ speed-up and projection reduction together with the remaining miss probability. Timings come from the shared cumulative sweep; they are not seven independent reruns.

In [ ]:
cost_quality = summary[['k', 'exact_recovery_rate', 'miss_probability_upper_confidence', 'strict_runtime_ms_mean', 'strict_projections_mean', 'exact_runtime_ms_mean', 'exact_projections_mean']].copy()
cost_quality['speed-up vs exhaustive'] = cost_quality['exact_runtime_ms_mean'] / cost_quality['strict_runtime_ms_mean']
cost_quality['projection reduction'] = 1.0 - cost_quality['strict_projections_mean'] / cost_quality['exact_projections_mean']
display(cost_quality.style.format({
    'exact_recovery_rate': '{:.1%}', 'miss_probability_upper_confidence': '{:.2%}',
    'strict_runtime_ms_mean': '{:.2f}', 'strict_projections_mean': '{:.1f}',
    'exact_runtime_ms_mean': '{:.1f}', 'exact_projections_mean': '{:.1f}',
    'speed-up vs exhaustive': '{:.1f}x', 'projection reduction': '{:.1%}',
}))

fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.plot(cost_quality['strict_runtime_ms_mean'], 1.0 - cost_quality['exact_recovery_rate'], marker='o')
for row in cost_quality.itertuples():
    ax.annotate(f'k={row.k}', (row.strict_runtime_ms_mean, 1.0 - row.exact_recovery_rate), xytext=(5, 5), textcoords='offset points')
ax.set(xlabel='Mean strict-prefix time (ms)', ylabel='Empirical probability of missing $D^*$')
ax.set_yscale('log')
ax.grid(alpha=.25, which='both')
sns.despine()
fig.tight_layout()

## Worst-case queries

A small mean can conceal rare large errors. We identify the ten largest $k=1$ gaps and trace the same queries across the full sweep.

In [ ]:
worst_ids = queries.query('k == 1').nlargest(10, 'gap_absolute')['query_idx'].tolist()
worst = queries.loc[queries['query_idx'].isin(worst_ids)].copy()
worst_table = worst.pivot(index='query_idx', columns='k', values='gap_absolute')
worst_metadata = queries.query('k == 1').set_index('query_idx').loc[worst_ids, ['target_class', 'exact_distance', 'minimum_k_exact']]
display(worst_metadata.join(worst_table.add_prefix('gap k=')).sort_values('gap k=1', ascending=False).style.format(precision=4))

fig, ax = plt.subplots(figsize=(7.0, 4.4))
for query_idx, group in worst.groupby('query_idx'):
    ax.plot(group['k'], group['gap_absolute'], marker='o', alpha=.75, label=str(query_idx))
ax.set(xlabel='k', ylabel='$D_k-D^*$', xticks=expected_k)
ax.grid(alpha=.25)
ax.legend(title='Query', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, ncol=2)
sns.despine()
fig.tight_layout()

## Held-out conformal calibration of $k$

To illustrate the proposed statistical calibration, the 1,000 query ranks are deterministically split into 500 calibration and 500 evaluation queries. For miscoverage level $\alpha$, we select the $\lceil(n+1)(1-\alpha)\rceil$-th ordered calibration rank. Evaluation coverage is reported only as an independent diagnostic; the conformal guarantee itself is marginal under exchangeability. A selected value `>7` would be unresolved because the current sweep right-censors those ranks.

In [ ]:
rng = np.random.default_rng(42)
permutation = rng.permutation(len(ranks))
calibration = ranks.iloc[permutation[:500]].copy()
evaluation = ranks.iloc[permutation[500:]].copy()
calibration_ranks = np.sort(calibration['recovery_rank_censored'].to_numpy(int))
conformal_records = []
for alpha in [0.10, 0.05, 0.025, 0.01]:
    order = int(np.ceil((len(calibration_ranks) + 1) * (1.0 - alpha)))
    if order > len(calibration_ranks):
        calibrated_k = 8
    else:
        calibrated_k = int(calibration_ranks[order - 1])
    resolved = calibrated_k <= max(expected_k)
    if resolved:
        covered = int((evaluation['recovery_rank_censored'] <= calibrated_k).sum())
        lower, upper, _ = clopper_pearson(covered, len(evaluation))
        empirical_coverage = covered / len(evaluation)
    else:
        empirical_coverage = lower = upper = np.nan
    conformal_records.append({
        'alpha': alpha, 'target coverage': 1 - alpha,
        'calibrated k': calibrated_k if resolved else '>7',
        'evaluation coverage': empirical_coverage,
        'evaluation CI lower': lower, 'evaluation CI upper': upper,
    })
conformal_results = pd.DataFrame(conformal_records)
conformal_results.style.format({
    'alpha': '{:.1%}', 'target coverage': '{:.1%}', 'evaluation coverage': '{:.1%}',
    'evaluation CI lower': '{:.1%}', 'evaluation CI upper': '{:.1%}',
})

## Interpretation checklist

- Report strict validity and exact-optimum recovery as different quantities.
- Inspect the tail and maximum of $D_k-D^*$, not only its mean.
- Report both empirical probabilities and finite-sample confidence bounds.
- Use paired comparisons because all $k$ values share the same queries.
- Treat `>7` recovery ranks as right-censored; extend the sweep before claiming a plateau.
- The confidence and conformal statements concern optimality over this fixed certified atlas and the query distribution, not the network's full target-class preimage or unseen architectures.